Import Libraries

In [25]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

Upload Your PDF

In [26]:
from google.colab import files
uploaded = files.upload()

Saving Medical_book.pdf to Medical_book (1).pdf


In [27]:
!mkdir Data
!mv Medical_book.pdf Data/

Load the PDF

In [28]:
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

extracted_data = load_pdf_file("Data/")
print(len(extracted_data))

637


Split Text into Chunks

In [29]:
def text_split(extracted_data):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


text_chunks = text_split(extracted_data)

print("Number of Chunks:", len(text_chunks))

Number of Chunks: 5860


Install Embedding Model

In [31]:
!pip install langchain-huggingface

Create Embeddings


In [32]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings


embeddings = download_hugging_face_embeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test Embedding

In [33]:
query_result = embeddings.embed_query("Hello world")
print("Embedding length:", len(query_result))

Embedding length: 384


Create Vector Database (FAISS — Easier than Pinecone)

In Colab it is much easier to use FAISS instead of Pinecone.

In [34]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.5 MB/s eta 0:00:00


In [35]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    text_chunks,
    embeddings
)

Create Retriever

In [36]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

Test Retrieval

In [37]:
docs = retriever.invoke("What is acne?")
docs

[Document(id='0f65d027-02e9-4eb7-94e2-d2195bb3a15b', metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'Data/Medical_book.pdf', 'total_pages': 637, 'page': 39, 'page_label': '40'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='214991ef-4eec-4c72-8e27-d7dd47a4eb3e', metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'Data/Medical_book.pdf', 'total_pages': 637, 'page': 38, 'page_label': '39'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 

LLM RESPONSE:

In [45]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.1 MB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(
    api_key="sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Explain what TB is in simple terms"}
    ]
)

print(response.choices[0].message.content)

TB, also known as Tuberculosis, is a type of bacterial infection that mainly affects the lungs. It's usually spread through the air when an infected person coughs, sneezes, or talks.

Imagine tiny germs called Mycobacterium tuberculosis (M. tuberculosis) entering your body through your nose or mouth. They then multiply and cause damage to your lung tissue. This can lead to symptoms like:

- Prolonged coughing (often with blood)
- Coughing up mucus or sputum
- Chest pain or discomfort
- Fatigue (feeling very tired)
- Weight loss

TB can spread to other parts of your body, including your kidneys, spine, and brain, if left untreated. However, it's highly curable with antibiotics if caught early.

There are two main types of TB:

1. **Pulmonary TB** (TB in the lungs): the most common type
2. **Extrapulmonary TB** (TB in the other parts of the body): less common, but can be just as serious

TB is more common in certain groups, such as:

- People living in crowded or poor areas
- People with